# Span Alignment Diagnostics

Two diagnostic sections:
1. **10 worked examples** — shows the raw output of `find_parent_token` so you can see exactly what the algorithm produces.
2. **QED-train statistics** — iterates all 7 618 samples and reports span-count distribution (mean / max / min / histogram).

In [ ]:
import os

HOME = '/workspace'
print(HOME)

# Tokenizer paths — adjust if your local model directory is different.
# These match the student/teacher pair used in the QED distillation runs.
STUDENT_TOKENIZER = f"{HOME}/Multi-Level-OT/EleutherAI/opt-350m"  
TEACHER_TOKENIZER = f"{HOME}/models/Llama-2-7b-chat-hf"
  

QED_DISK_PATH = f"{HOME}/Multi-Level-OT/llm_distillation/datasets/processed/qed"

N_EXAMPLES = 10

In [ ]:
#Imports 
import sys
import textwrap
import numpy as np
from collections import Counter

sys.path.insert(0, f"{HOME}/Multi-Level-OT")

from datasets import load_from_disk
from transformers import AutoTokenizer
from train.span_match import find_parent_token

print("Imports OK")

In [ ]:

student_tok = AutoTokenizer.from_pretrained(STUDENT_TOKENIZER, use_fast=True)
teacher_tok = AutoTokenizer.from_pretrained(TEACHER_TOKENIZER, use_fast=True)

qed = load_from_disk(QED_DISK_PATH)["train"]
print(f"QED train split: {len(qed)} samples")
print(f"Columns: {qed.column_names}")
print(f"\nExample answer field: {qed[0]['original_nq_answers'][0]['string']!r}")

In [ ]:

def get_offsets(text: str, tokenizer) -> list:
    """Return [(char_start, char_end), ...] filtering out (0,0) special tokens."""
    if not text or not text.strip():
        return []
    try:
        enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    except Exception:
        return []
    return [(int(s), int(e)) for s, e in enc["offset_mapping"] if s != e]


def answer_text(sample: dict) -> str:
    """Pull the first gold answer string from a QED sample."""
    answers = sample.get("original_nq_answers", [])
    if answers and isinstance(answers[0], dict):
        return answers[0].get("string", "")
    return ""


def span_count(sample: dict) -> int:
    """Number of parent spans found for a sample (-1 if alignment fails)."""
    text = answer_text(sample)
    s_off = get_offsets(text, student_tok)
    t_off = get_offsets(text, teacher_tok)
    if not s_off or not t_off:
        return -1
    s_dict, _ = find_parent_token(s_off, t_off)
    return len(s_dict)

print("Helpers defined")

---
## Part 1 — Worked Examples

For each example we print:
- The gold answer text
- How the student tokenizer splits it (with character offsets)
- How the teacher tokenizer splits it (with character offsets)
- The parent spans — minimal char ranges respected by **both** tokenisations
- Which token indices from each side belong to each span

In [ ]:
DIVIDER = "=" * 70
SUB     = "-" * 60

def fmt_tokens(text, offsets, indices):
    """Return a compact string showing selected tokens and their substrings."""
    parts = []
    for i in indices:
        if i < len(offsets):
            s, e = offsets[i]
            parts.append(f"[{i}]'{text[s:e]}'")
    return "  ".join(parts) if parts else "(none)"


printed = 0
idx = 0

while printed < N_EXAMPLES and idx < len(qed):
    sample = qed[idx]
    idx += 1
    text = answer_text(sample)
    if not text:
        continue

    s_off = get_offsets(text, student_tok)
    t_off = get_offsets(text, teacher_tok)
    if not s_off or not t_off:
        continue

    s_dict, t_dict = find_parent_token(s_off, t_off)
    if not s_dict:
        continue

    print(DIVIDER)
    print(f"Example {printed + 1}  (QED index {idx - 1})")
    print(f"Answer text  : {text!r}")
    print()

    print("Student tokens (index → char-range → substring):")
    s_parts = [f"[{i}] {s_off[i]} '{text[s_off[i][0]:s_off[i][1]]}'"
               for i in range(len(s_off))]
    print("  " + "   ".join(s_parts))
    print()

    print("Teacher tokens (index → char-range → substring):")
    t_parts = [f"[{i}] {t_off[i]} '{text[t_off[i][0]:t_off[i][1]]}'"
               for i in range(len(t_off))]
    print("  " + "   ".join(t_parts))
    print()

    print(f"Parent spans ({len(s_dict)} total):")
    for span_id, (span_key, s_indices) in enumerate(s_dict.items()):
        t_indices = t_dict.get(span_key, [])
        char_slice = text[span_key[0]:span_key[1]]
        print(f"  Span {span_id}  chars {span_key}  '{char_slice}'")
        print(f"    student → {fmt_tokens(text, s_off, s_indices)}")
        print(f"    teacher → {fmt_tokens(text, t_off, t_indices)}")
    print()

    printed += 1

print(DIVIDER)
print(f"Done — printed {printed} examples.")

---
## Part 2 — QED-train Statistics

Iterate all 7 618 samples and collect:
- Number of parent spans per sample
- Samples where alignment fails (empty answer or tokenizer has no offsets) → `span_count = -1`

In [ ]:
from tqdm.auto import tqdm

counts = []           # span count per sample (aligned samples only)
n_failed = 0          # samples where alignment returned nothing
n_empty_answer = 0    # samples with empty answer text

for sample in tqdm(qed, desc="Scanning QED train"):
    text = answer_text(sample)
    if not text:
        n_empty_answer += 1
        continue

    s_off = get_offsets(text, student_tok)
    t_off = get_offsets(text, teacher_tok)
    if not s_off or not t_off:
        n_failed += 1
        continue

    s_dict, _ = find_parent_token(s_off, t_off)
    counts.append(len(s_dict))

counts = np.array(counts)
print("Done.")

In [ ]:
n_total     = len(qed)
n_aligned   = len(counts)

print("=" * 50)
print("QED train — span alignment statistics")
print("=" * 50)
print(f"Total samples          : {n_total}")
print(f"  empty answer         : {n_empty_answer}")
print(f"  no offsets / failed  : {n_failed}")
print(f"  successfully aligned : {n_aligned}")
print()
if n_aligned > 0:
    print(f"Spans per sample (aligned only):")
    print(f"  mean   : {counts.mean():.2f}")
    print(f"  median : {np.median(counts):.1f}")
    print(f"  std    : {counts.std():.2f}")
    print(f"  min    : {counts.min()}")
    print(f"  max    : {counts.max()}")
    print(f"  p25    : {np.percentile(counts, 25):.1f}")
    print(f"  p75    : {np.percentile(counts, 75):.1f}")
    print(f"  p95    : {np.percentile(counts, 95):.1f}")
    print()
    freq = Counter(counts.tolist())
    print("Span-count frequency (top 15 values):")
    for k, v in sorted(freq.items())[:15]:
        bar = "#" * min(40, v * 40 // n_aligned)
        print(f"  {k:3d} spans : {v:5d} samples ({v/n_aligned*100:5.1f}%)  {bar}")
else:
    print("No aligned samples — check tokenizer paths.")